In [13]:
from dotenv import load_dotenv
from langfuse import Langfuse
from langfuse import get_client, observe,Evaluation
import time
import os

In [4]:
load_dotenv()

True

In [5]:
import sys
import os

# Agrega la carpeta raíz del proyecto al sys.path
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

print(f"Ruta raíz agregada con éxito: {root_path}")

from agent.nodes import (intent_analyzer_node)


Ruta raíz agregada con éxito: d:\Users\juanp_schamun\Documents\GitRepositories\BitoviApp\src


In [7]:
import time
from langfuse import get_client
from langchain_core.messages import HumanMessage


# 1. Inicialización
langfuse = get_client()
DATASET_NAME = "intent_analyzer_structured_set"
EXPERIMENT_RUN_NAME = f"intent_v1_clean_{int(time.time())}"

# 2. El Wrapper de evaluación (Solo prepara la entrada y evalúa la salida)
def evaluar_intent_task(*, item, **kwargs):
    raw_message_text = item.input["messages"][-1][1]
    
    # Preparamos el estado simulado tal cual lo espera tu nodo
    estado_simulado = {
        "messages": [HumanMessage(content=raw_message_text)]
    }
    
    # Ejecutamos TU NODO REAL directamente
    resultado_nodo = intent_analyzer_node(estado_simulado)
    
    # Extraemos los datos limpios de la salida de tu nodo
    intent_detectado = resultado_nodo.get("task_type")
    reasoning_llm = resultado_nodo.get("intent_reasoning", "")
    
    # Si el intent viene como un Enum de Pydantic, extraemos su string puro
    if hasattr(intent_detectado, 'value'):
        intent_detectado = intent_detectado.value
        
    intent_esperado = item.expected_output.get("intent")
    
    # Función auxiliar para comparar sin rompernos por mayúsculas/minúsculas o namespaces
    def normalizar(texto):
        return str(texto).split(".")[-1].lower().strip() if texto else ""
    
    es_correcto = 1 if normalizar(intent_detectado) == normalizar(intent_esperado) else 0
    
    # Guardamos el puntaje en la traza del experimento
    if "run_item" in kwargs:
        kwargs["run_item"].score(
            name="exact_match_intent",
            value=es_correcto,
            comment=f"Pregunta: {raw_message_text}\n"
                    f"Esperado: {intent_esperado} | Obtenido: {intent_detectado}\n"
                    f"Razonamiento: {reasoning_llm}"
        )
        
    return {
        "intent_detectado": intent_detectado,
        "reasoning": reasoning_llm
    }

# 3. Lanzamos el experimento
dataset = langfuse.get_dataset(DATASET_NAME)
print(f"🚀 Corriendo experimento limpio: '{EXPERIMENT_RUN_NAME}'...")

result = dataset.run_experiment(
    name=EXPERIMENT_RUN_NAME,
    description="Evaluación del Intent Analyzer consumiendo las salidas del estado",
    task=evaluar_intent_task
)

print("\n🏁 ¡Experimento finalizado con éxito!")
print(result.format())

🚀 Corriendo experimento limpio: 'intent_v1_clean_1779199044'...

[INTENT_ANALIZER-NODE] Clasificando Intención del Usuario
[INTENT_ANALIZER-NODE] Type: TaskType.LISTING | Reason: The user wants to see a list of available resources or documents, blogs, articles or posts. The keyword 'last' indicates they want the most recent one.

[INTENT_ANALIZER-NODE] Clasificando Intención del Usuario
[INTENT_ANALIZER-NODE] Type: TaskType.SINTESIS | Reason: The user is asking for a definition of a technical term, which falls under the SINTESIS category. The goal is to understand what Retrieval-Augmented Generation (RAG) means.

[INTENT_ANALIZER-NODE] Clasificando Intención del Usuario
[INTENT_ANALIZER-NODE] Type: TaskType.LISTING | Reason: The user wants to see a list of available resources or documents, blogs, articles, or posts related to 'articles' or 'testing' for the year '2024'. The goal is to retrieve the document containers.

[INTENT_ANALIZER-NODE] Clasificando Intención del Usuario
[INTENT_A

In [14]:
import time
from langfuse import get_client, Evaluation  # <--- Importante sumar Evaluation acá
from langchain_core.messages import HumanMessage

# 1. Inicialización
langfuse = get_client()
DATASET_NAME = "intent_analyzer_structured_set"
EXPERIMENT_RUN_NAME = f"intent_v2_decoupled_{int(time.time())}"


# 2. El Evaluador Independiente (Solo compara outputs vs expected y devuelve el Score)
def exact_match_evaluator(output, expected_output, **kwargs) -> Evaluation:
    # Helper para normalizar strings
    def normalizar(texto):
        return str(texto).split(".")[-1].lower().strip() if texto else ""
        
    intent_detectado = output.get("intent_detectado")
    intent_esperado = expected_output.get("intent")
    reasoning_llm = output.get("reasoning", "")
    
    # Medición de correctitud
    es_correcto = 1 if normalizar(intent_detectado) == normalizar(intent_esperado) else 0
    
    # Retornamos el objeto Evaluation que Langfuse mapea automáticamente a la columna Scores
    return Evaluation(
        name="exact_match_intent",
        value=es_correcto,
        comment=f"Esperado: {intent_esperado} | Obtenido: {intent_detectado}\n"
                f"Razonamiento: {reasoning_llm}"
    )


# 3. La Tarea (Queda 100% limpia de lógica de scoring)
def evaluar_intent_task(*, item, **kwargs):
    raw_message_text = item.input["messages"][-1][1]
    
    # Preparamos el estado simulado tal cual lo espera tu nodo
    estado_simulado = {
        "messages": [HumanMessage(content=raw_message_text)]
    }
    
    # Ejecutamos TU NODO REAL directamente
    resultado_nodo = intent_analyzer_node(estado_simulado)
    
    # Extraemos los datos limpios de la salida de tu nodo (Pydantic/Dict)
    intent_detectado = resultado_nodo.get("task_type")
    reasoning_llm = resultado_nodo.get("intent_reasoning", "")
    
    # Si el intent viene como un Enum, extraemos su string puro
    if hasattr(intent_detectado, 'value'):
        intent_detectado = intent_detectado.value
        
    # Retornamos la salida limpia de la predicción. 
    # Este diccionario es el que va a recibir el evaluador en el parámetro 'output'
    return {
        "intent_detectado": intent_detectado,
        "reasoning": reasoning_llm
    }


# 4. Lanzamos el experimento pasándole la lista de evaluadores
dataset = langfuse.get_dataset(DATASET_NAME)
print(f"🚀 Corriendo experimento desacoplado: '{EXPERIMENT_RUN_NAME}'...")

result = dataset.run_experiment(
    name=EXPERIMENT_RUN_NAME,
    description="Evaluación del Intent Analyzer usando evaluadores independientes",
    task=evaluar_intent_task,
    evaluators=[exact_match_evaluator]  # <--- El SDK ejecuta esto en paralelo por cada ítem
)

print("\n🏁 ¡Experimento finalizado con éxito!")

# 5. Forzar la subida de los datos de los hilos secundarios
langfuse.flush()


🚀 Corriendo experimento desacoplado: 'intent_v2_decoupled_1779201619'...

[INTENT_ANALIZER-NODE] Clasificando Intención del Usuario
[INTENT_ANALIZER-NODE] Type: TaskType.SINTESIS | Reason: The user wants to understand the topic of the last article, which is a content-based question.

[INTENT_ANALIZER-NODE] Clasificando Intención del Usuario
[INTENT_ANALIZER-NODE] Type: TaskType.SINTESIS | Reason: The user wants to understand what a vector store is, which is a conceptual definition.

[INTENT_ANALIZER-NODE] Clasificando Intención del Usuario
[INTENT_ANALIZER-NODE] Type: TaskType.SINTESIS | Reason: The user is asking for a definition of a technical term, which falls under the SINTESIS category. The goal is to understand what Retrieval-Augmented Generation (RAG) means.

[INTENT_ANALIZER-NODE] Clasificando Intención del Usuario
[INTENT_ANALIZER-NODE] Type: TaskType.SINTESIS | Reason: The user wants to understand the information inside a document, specifically the topic of the latest article